# Recurrent Architectures: RNNs, LSTMs, GRUs & Bidirectional Sequence Modeling

This interactive lab covers:
1. **Recurrent Gating Mechanisms from Scratch**: Explicit equations for `SimpleRNNCell`, `LSTMCell`, and `GRUCell`.
2. **Vanishing Gradient Analysis**: Empirical measurement of gradient attenuation through Backpropagation Through Time (BPTT).
3. **Bidirectional LSTM Classifier**: Processing temporal sequences in forward and backward directions.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import torch
import torch.nn as nn
from rnn_engine import (
    ScratchSimpleRNNCell, ScratchLSTMCell, ScratchGRUCell,
    BidirectionalLSTMClassifier, measure_bptt_gradient_norms
)

print('Recurrent Neural Network modules loaded successfully!')

## 1. Scratch Recurrent Transitions (LSTM & GRU Gating)

We pass an input vector sequence through our manually implemented `ScratchLSTMCell` and `ScratchGRUCell`.

In [ ]:
torch.manual_seed(42)
batch_size = 3
in_dim = 4
h_dim = 8

lstm_cell = ScratchLSTMCell(input_size=in_dim, hidden_size=h_dim)
gru_cell = ScratchGRUCell(input_size=in_dim, hidden_size=h_dim)

x_t = torch.randn(batch_size, in_dim)
h_prev = torch.zeros(batch_size, h_dim)
c_prev = torch.zeros(batch_size, h_dim)

h_lstm, c_lstm = lstm_cell.forward(x_t, h_prev, c_prev)
h_gru = gru_cell.forward(x_t, h_prev)

print(f'LSTM Hidden State Shape: {h_lstm.shape} | Cell State Shape: {c_lstm.shape}')
print(f'GRU  Hidden State Shape: {h_gru.shape}')
assert h_lstm.shape == (batch_size, h_dim)

## 2. Vanishing Gradient Diagnostic (Plain RNN vs. LSTM)

We compute gradient norms of the loss with respect to early inputs across 15 timesteps to verify the gradient highway effect.

In [ ]:
rnn_grads, lstm_grads = measure_bptt_gradient_norms(seq_len=15, hidden_dim=32)

print(f'{"Timestep (t)":<15} | {"Plain RNN Grad Norm":<22} | {"LSTM Grad Norm":<20}')
print('-' * 62)
for t in range(15):
    print(f'{t:<15} | {rnn_grads[t]:<22.6f} | {lstm_grads[t]:<20.6f}')

lstm_retention = lstm_grads[0] / (lstm_grads[-1] + 1e-8)
rnn_retention = rnn_grads[0] / (rnn_grads[-1] + 1e-8)
print(f'\nGradient retention ratio (t=0 / t=14):')
print(f'  Plain RNN: {rnn_retention:.2%}')
print(f'  LSTM:      {lstm_retention:.2%}')

## 3. Production Bidirectional LSTM Classifier

We test sequence classification on a batch of token sequences using `BidirectionalLSTMClassifier`.

In [ ]:
model = BidirectionalLSTMClassifier(
    vocab_size=500,
    embed_dim=32,
    hidden_dim=64,
    num_classes=3,
    num_layers=2,
    dropout=0.2
)

sample_batch = torch.randint(0, 500, (8, 25))  # Batch=8, SeqLen=25
logits = model(sample_batch)

print(f'Input Batch Shape:  {sample_batch.shape}')
print(f'Output Logits Shape: {logits.shape}')
assert logits.shape == (8, 3), 'Incorrect output logits shape!'